In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, UpSampling2D, Concatenate
)
from sklearn.utils import shuffle
from pathlib import Path

# --- CONFIG ---
train_csv = Path("/path/to/training/data.csv")
input_shape = (128, 128, 3)
batch_size = 32
epochs = 50
save_path = "unet2d_regression_model_Q1.keras"

# --- Data Generator ---
class SpectrogramDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, dataframe, batch_size, input_shape, shuffle=True):
        self.df = dataframe.reset_index(drop=True)
        self.batch_size = batch_size
        self.input_shape = input_shape
        self.shuffle = shuffle
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, index):
        batch = self.df.iloc[index * self.batch_size:(index + 1) * self.batch_size]
        X = np.zeros((len(batch), *self.input_shape), dtype=np.float32)
        y = np.zeros((len(batch),), dtype=np.float32)

        for i, (_, row) in enumerate(batch.iterrows()):
            try:
                arr = np.load(row["filepath"]).astype(np.float32)
                arr = tf.image.resize(arr[..., np.newaxis], self.input_shape[:2]).numpy()
                arr = np.repeat(arr, 3, axis=-1)  # Convert grayscale to RGB
                X[i] = arr
                y[i] = row["label"]
            except Exception as e:
                print(f"⚠️ Error loading {row['filepath']}: {e}")

        return X, y

    def on_epoch_end(self):
        if self.shuffle:
            self.df = shuffle(self.df)

# --- U-Net Building Blocks ---
def conv_block(x, filters):
    x = Conv2D(filters, (3, 3), padding='same', activation='relu')(x)
    x = Conv2D(filters, (3, 3), padding='same', activation='relu')(x)
    return x

def unet_regression(input_shape=(128, 128, 3)):
    inputs = Input(shape=input_shape)

    # Encoder
    c1 = conv_block(inputs, 64)
    p1 = MaxPooling2D((2, 2))(c1)

    c2 = conv_block(p1, 128)
    p2 = MaxPooling2D((2, 2))(c2)

    # Bottleneck
    b = conv_block(p2, 512)

    # Decoder
    u1 = UpSampling2D((2, 2))(b)
    m1 = Concatenate()([u1, c2])
    c3 = conv_block(m1, 128)

    u2 = UpSampling2D((2, 2))(c3)
    m2 = Concatenate()([u2, c1])
    c4 = conv_block(m2, 64)

    # Output layer for regression
    outputs = Conv2D(1, (1, 1), activation='linear')(c4)

    model = Model(inputs, outputs)
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])

    return model

# --- Load Data ---
train_df = pd.read_csv(train_csv)
train_gen = SpectrogramDataGenerator(train_df, batch_size, input_shape)

# --- Train U-Net Model ---
model = unet_regression(input_shape=input_shape)
model.summary()

checkpoint = tf.keras.callbacks.ModelCheckpoint(save_path, save_best_only=False)

model.fit(
    train_gen,
    epochs=epochs,
    callbacks=[checkpoint],
    verbose=1
)

print(f"\n✅ U-Net 2D regression model trained and saved as: {save_path}")


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import mean_squared_error
import tensorflow as tf

# --- Configuration ---
val_csv = Path("/path/to/validation/data.csv")
input_shape = (128, 128, 3)
batch_size = 32
model_path = "/path/to/trained/unet2d_regression_model_Q1.keras"  # Adjust if you saved as .keras

# --- Reload Validation Data Generator ---
class SpectrogramDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, dataframe, batch_size, input_shape, shuffle=False):
        self.df = dataframe.reset_index(drop=True)
        self.batch_size = batch_size
        self.input_shape = input_shape
        self.shuffle = shuffle

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, index):
        batch = self.df.iloc[index * self.batch_size:(index + 1) * self.batch_size]
        X, y = [], []

        for _, row in batch.iterrows():
            try:
                arr = np.load(row["filepath"]).astype(np.float32)
                arr = tf.image.resize(arr[..., np.newaxis], self.input_shape[:2]).numpy()
                arr = np.repeat(arr, 3, axis=-1)
                X.append(arr)
                y.append(row["label"])
            except Exception as e:
                print(f"⚠️ Skipping {row['filepath']}: {e}")

        if not X:
            return np.array([]), np.array([])
        return np.stack(X), np.array(y, dtype=np.float32)

# --- Load Model and Data ---
val_df = pd.read_csv(val_csv)
val_gen = SpectrogramDataGenerator(val_df, batch_size, input_shape)
model = tf.keras.models.load_model(model_path)

# --- Get Predictions Safely ---
y_true, y_pred = [], []

for i in range(len(val_gen)):
    X_batch, y_batch = val_gen[i]

    if len(X_batch) == 0:
        continue  # skip empty batches

    preds = model.predict(X_batch, verbose=0)

    # Handle output shape (for regression models that return images)
    if preds.ndim == 4 and preds.shape[-1] == 1:
        preds = preds.mean(axis=(1, 2, 3))  # flatten to scalar per sample
    elif preds.ndim == 2:
        preds = preds[:, 0]

    y_true.extend(y_batch)
    y_pred.extend(preds)

# --- Compute MSE if we got predictions ---
if y_true and y_pred:
    mse = mean_squared_error(y_true, y_pred)
    print(f"📊 Mean Squared Error (Validation): {mse:.4f}")
else:
    print("⚠️ No valid validation data was found to compute MSE.")


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import mean_squared_error
import tensorflow as tf

# --- Configuration ---
train_csv = Path("/path/to/training/data.csv")
input_shape = (128, 128, 3)
batch_size = 32
model_path = "/path/to/trained/unet2d_regression_model_Q1.keras"  # or .keras

# --- Data Generator ---
class SpectrogramDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, dataframe, batch_size, input_shape, shuffle=False):
        self.df = dataframe.reset_index(drop=True)
        self.batch_size = batch_size
        self.input_shape = input_shape
        self.shuffle = shuffle

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, index):
        batch = self.df.iloc[index * self.batch_size:(index + 1) * self.batch_size]
        X, y = [], []

        for _, row in batch.iterrows():
            try:
                arr = np.load(row["filepath"]).astype(np.float32)
                arr = tf.image.resize(arr[..., np.newaxis], self.input_shape[:2]).numpy()
                arr = np.repeat(arr, 3, axis=-1)
                X.append(arr)
                y.append(row["label"])
            except Exception as e:
                print(f"⚠️ Skipping {row['filepath']}: {e}")

        if not X:
            return np.array([]), np.array([])
        return np.stack(X), np.array(y, dtype=np.float32)

# --- Load Model and Data ---
train_df = pd.read_csv(train_csv)
train_gen = SpectrogramDataGenerator(train_df, batch_size, input_shape)
model = tf.keras.models.load_model(model_path)

# --- Predict and Compute MSE ---
y_true, y_pred = [], []

for i in range(len(train_gen)):
    X_batch, y_batch = train_gen[i]

    if len(X_batch) == 0:
        continue

    preds = model.predict(X_batch, verbose=0)

    # Handle 2D/3D predictions for regression maps
    if preds.ndim == 4 and preds.shape[-1] == 1:
        preds = preds.mean(axis=(1, 2, 3))  # mean per image
    elif preds.ndim == 2:
        preds = preds[:, 0]

    y_true.extend(y_batch)
    y_pred.extend(preds)

# --- Compute and Print MSE ---
if y_true and y_pred:
    mse = mean_squared_error(y_true, y_pred)
    print(f"📊 Mean Squared Error (Training): {mse:.4f}")
else:
    print("⚠️ No valid training data was found to compute MSE.")


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import mean_squared_error
import tensorflow as tf

# --- Configuration ---
test_csv = Path("/path/to/testing/data.csv")
input_shape = (128, 128, 3)
batch_size = 32
model_path = "/path/to/trained/unet2d_regression_model_Q1.keras"  # or .keras

# --- Data Generator ---
class SpectrogramDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, dataframe, batch_size, input_shape, shuffle=False):
        self.df = dataframe.reset_index(drop=True)
        self.batch_size = batch_size
        self.input_shape = input_shape
        self.shuffle = shuffle

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, index):
        batch = self.df.iloc[index * self.batch_size:(index + 1) * self.batch_size]
        X, y = [], []

        for _, row in batch.iterrows():
            try:
                arr = np.load(row["filepath"]).astype(np.float32)
                arr = tf.image.resize(arr[..., np.newaxis], self.input_shape[:2]).numpy()
                arr = np.repeat(arr, 3, axis=-1)
                X.append(arr)
                y.append(row["label"])
            except Exception as e:
                print(f"⚠️ Skipping {row['filepath']}: {e}")

        if not X:
            return np.array([]), np.array([])
        return np.stack(X), np.array(y, dtype=np.float32)

# --- Load Model and Data ---
test_df = pd.read_csv(test_csv)
test_gen = SpectrogramDataGenerator(test_df, batch_size, input_shape)
model = tf.keras.models.load_model(model_path)

# --- Predict and Compute RMSE ---
y_true, y_pred = [], []

for i in range(len(test_gen)):
    X_batch, y_batch = test_gen[i]

    if len(X_batch) == 0:
        continue

    preds = model.predict(X_batch, verbose=0)

    # Handle 2D/3D predictions for regression maps
    if preds.ndim == 4 and preds.shape[-1] == 1:
        preds = preds.mean(axis=(1, 2, 3))  # mean per image
    elif preds.ndim == 2:
        preds = preds[:, 0]

    y_true.extend(y_batch)
    y_pred.extend(preds)

# --- Compute and Print RMSE ---
if y_true and y_pred:
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    print(f"📊 Root Mean Squared Error (Testing): {rmse:.4f}")
else:
    print("⚠️ No valid test data was found to compute RMSE.")


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import r2_score
import tensorflow as tf

# --- Configuration ---
test_csv = Path("/path/to/testing/data.csv")
input_shape = (128, 128, 3)
batch_size = 32
model_path = "/path/to/trained/unet2d_regression_model_Q1.keras"  # or .keras

# --- Data Generator ---
class SpectrogramDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, dataframe, batch_size, input_shape, shuffle=False):
        self.df = dataframe.reset_index(drop=True)
        self.batch_size = batch_size
        self.input_shape = input_shape
        self.shuffle = shuffle

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, index):
        batch = self.df.iloc[index * self.batch_size:(index + 1) * self.batch_size]
        X, y = [], []

        for _, row in batch.iterrows():
            try:
                arr = np.load(row["filepath"]).astype(np.float32)
                arr = tf.image.resize(arr[..., np.newaxis], self.input_shape[:2]).numpy()
                arr = np.repeat(arr, 3, axis=-1)
                X.append(arr)
                y.append(row["label"])
            except Exception as e:
                print(f"⚠️ Skipping {row['filepath']}: {e}")

        if not X:
            return np.array([]), np.array([])
        return np.stack(X), np.array(y, dtype=np.float32)

# --- Load Model and Data ---
test_df = pd.read_csv(test_csv)
test_gen = SpectrogramDataGenerator(test_df, batch_size, input_shape)
model = tf.keras.models.load_model(model_path)

# --- Predict and Compute R² ---
y_true, y_pred = [], []

for i in range(len(test_gen)):
    X_batch, y_batch = test_gen[i]

    if len(X_batch) == 0:
        continue

    preds = model.predict(X_batch, verbose=0)

    # Handle 2D/3D predictions for regression maps
    if preds.ndim == 4 and preds.shape[-1] == 1:
        preds = preds.mean(axis=(1, 2, 3))
    elif preds.ndim == 2:
        preds = preds[:, 0]

    y_true.extend(y_batch)
    y_pred.extend(preds)

# --- Compute and Print R² ---
if y_true and y_pred:
    r2 = r2_score(y_true, y_pred)
    print(f"📈 Coefficient of Determination (R²) on Testing Data: {r2:.4f}")
else:
    print("⚠️ No valid test data was found to compute R².")
